# Carga Criolla S.A. — simulación concurrente

**Programación Concurrente — UNLaM — Actividad Práctica: Comunicación y Sincronización**

## El problema

Carga Criolla S.A. transporta bolsones de harina de **Tapiales (Bs. As.)** a **Fernández (Sgo. del Estero)**
y bolsas de carbón en el sentido inverso. La empresa quiere saber **cuántos camiones (N) le conviene asignar**
para cubrir **X viajes** en el menor tiempo posible.

Restricciones operativas del relevamiento:

| Regla del enunciado | Consecuencia para el programa |
|---|---|
| Cada planta maneja como máximo 2 camiones a la vez: uno cargando y otro descargando | Cada planta tiene 2 recursos de capacidad 1 |
| El primer camión disponible ingresa a cargar *si la zona está libre* | Hay que **bloquear** al camión hasta que se libere la zona |
| En Fernández se reposta antes de partir; hay **2 surtidores** de Diesel | Recurso compartido de capacidad 2 |
| El viaje dura entre 18 y 24 h *según el chofer* | Cada camión tiene su propia velocidad (no determinismo) |
| Carga y descarga: 2 h. Recarga de combustible: 1 h | Duraciones fijas |
| Los N camiones arrancan en Bs. As. y la simulación termina cuando no quedan viajes | Condición de terminación compartida |

## Por qué esto es un problema de concurrencia

Los camiones **existen al mismo tiempo** y avanzan de forma independiente, pero **compiten por recursos
compartidos y limitados** (las zonas de carga y descarga, los surtidores, la bolsa de viajes pendientes).
Ese es exactamente el escenario que resuelven la **comunicación** (cómo se pasan información los flujos de
ejecución) y la **sincronización** (cómo se coordinan para no pisarse).

Un programa secuencial podría calcular el resultado con una fórmula, pero no *mostraría* lo que pide el
enunciado: el estado de cada camión mientras espera, carga o viaja.

## 1. Mapa del problema a primitivas de concurrencia

Antes de escribir código, decidimos qué es cada cosa. Este es el diseño completo:

| Elemento del dominio | Abstracción concurrente | Primitiva de Python |
|---|---|---|
| Camión / chofer | **Hilo** (flujo de ejecución dentro del mismo proceso) | `threading.Thread` |
| Zona de carga de una planta | Recurso exclusivo (capacidad 1) | `threading.Semaphore(1)` |
| Zona de descarga de una planta | Recurso exclusivo (capacidad 1) | `threading.Semaphore(1)` |
| Surtidores de Diesel en Fernández | Recurso limitado (capacidad 2) | `threading.BoundedSemaphore(2)` |
| Viajes pendientes en cada planta | Variable compartida mutable → **sección crítica** | `threading.Lock` (mutex) |
| Contador de viajes completados | Variable compartida mutable → **sección crítica** | `threading.Lock` (mutex) |
| Largada simultánea de los N camiones | Punto de encuentro | `threading.Barrier(N)` |
| Parte de operaciones / traza de eventos | **Paso de mensajes** productor–consumidor | `queue.Queue` + hilo monitor |
| Fin de la simulación | Espera de terminación | `Thread.join()` |

### Los dos modelos de comunicación, los dos presentes

1. **Memoria compartida**: las plantas, los semáforos y los contadores son objetos que *todos* los hilos ven.
   Es comunicación rápida pero **insegura por defecto**: exige exclusión mutua explícita.
2. **Paso de mensajes**: cada camión no imprime nada; **envía un evento** a una cola y sigue trabajando. Un
   único hilo monitor consume esa cola e imprime. La cola es el canal y el receptor es el dueño de la consola.

> **Por qué no usamos `print()` directo en los camiones:** `print` desde varios hilos entrelaza líneas
> (dos hilos escribiendo al mismo `stdout` es una carrera sobre un recurso compartido). Pasar los eventos
> por una cola convierte un recurso disputado en un recurso con **un solo dueño**, que es la forma más
> limpia de sincronizar: *el recurso no se comparte, se le pasa el trabajo a quien lo posee*.

## 2. Parámetros del dominio

Constantes tomadas del enunciado. Son de **sólo lectura** después de definidas: los datos inmutables se
pueden compartir entre hilos sin ninguna sincronización, porque nadie los escribe. Ésta es la primera regla
práctica de la concurrencia: **lo que no se muta, no necesita protección**.

In [11]:
import threading
import queue
import random
import time
import statistics

# --- Plantas ---
TAPIALES  = "Tapiales"    # Bs. As. -> se despachan bolsones de harina
FERNANDEZ = "Fernández"   # Sgo. del Estero -> se despachan bolsas de carbón

# --- Tiempos del enunciado, en horas simuladas ---
HORAS_CARGA       = 2.0    # igual en ambas plantas
HORAS_DESCARGA    = 2.0    # igual en ambas plantas
HORAS_COMBUSTIBLE = 1.0    # sólo en Fernández, antes de partir
VIAJE_MIN_H       = 18.0   # el tramo más rápido
VIAJE_MAX_H       = 24.0   # el tramo más lento
CANT_SURTIDORES   = 2      # la estación de servicio de Fernández

def formato_reloj(horas: float) -> str:
    # Convierte horas simuladas a "Día D HH:MM" para que la traza se lea como un parte de operaciones.
    dia = int(horas // 24) + 1
    resto = horas % 24
    return f"Día {dia} {int(resto):02d}:{int((resto % 1) * 60):02d}"

print("Parámetros cargados. Un viaje dura entre", VIAJE_MIN_H, "y", VIAJE_MAX_H, "horas.")

Parámetros cargados. Un viaje dura entre 18.0 y 24.0 horas.


## 3. El reloj simulado: escala de tiempo

No podemos esperar 20 horas reales por cada viaje. Usamos una **escala**: 1 hora simulada = `escala`
segundos reales. Con `escala = 0.05`, un viaje de 20 h tarda 1 segundo.

El punto conceptual importante es **qué hace `time.sleep()` dentro de un hilo**: pone a ese hilo en estado
*bloqueado*, libera la CPU y **suelta el GIL**. Mientras un camión "viaja", los demás hilos avanzan de
verdad. Por eso una simulación como ésta es genuinamente concurrente en Python a pesar del GIL: el
*Global Interpreter Lock* serializa la ejecución de bytecode, no las esperas.

> **Nota sobre el GIL:** en CPython un solo hilo ejecuta bytecode a la vez, así que los hilos **no** dan
> paralelismo real para cálculo intensivo (para eso se usa `multiprocessing`). Pero sí dan concurrencia
> real para tareas que esperan (E/S, `sleep`, red), que es justo el caso de camiones que tardan horas en
> llegar. Además, el GIL **no** vuelve seguras las operaciones compuestas: `x += 1` sigue siendo una
> condición de carrera, porque son varias instrucciones y el intérprete puede cambiar de hilo entre ellas.

In [12]:
class RelojSimulado:
    # Traduce horas simuladas a segundos reales y mide el tiempo transcurrido de la simulación.
    def __init__(self, escala: float):
        self.escala = escala
        self.t0 = time.perf_counter()

    def dormir(self, horas: float) -> None:
        # Bloquea SÓLO a este hilo: libera la CPU y el GIL, los demás camiones siguen avanzando.
        time.sleep(horas * self.escala)

    def ahora(self) -> float:
        # Horas simuladas transcurridas desde el arranque.
        return (time.perf_counter() - self.t0) / self.escala

## 4. Comunicación por paso de mensajes: el monitor de eventos

Cada camión es un **productor** de eventos; el monitor es el único **consumidor**. El canal es una
`queue.Queue`, que internamente ya está sincronizada (usa un `Lock` y variables de condición), así que
`put()` y `get()` son atómicos: **no hace falta que nosotros pongamos un candado alrededor**.

Esto implementa el patrón **productor–consumidor**, el más clásico de la comunicación entre hilos:

- Desacopla al productor del consumidor: el camión no espera a que se imprima nada.
- Serializa el acceso a `stdout` sin que ningún camión se bloquee contra otro.
- La terminación se resuelve con una **píldora de veneno** (*poison pill*): un valor centinela `None` que
  le dice al consumidor "no viene nada más, terminá". Es la forma estándar de cerrar un canal, porque
  respeta el orden: el centinela se encola *detrás* de todos los eventos ya publicados.

El monitor se crea como hilo **demonio** (`daemon=True`) para que nunca impida que el programa termine,
aunque de todos modos lo cerramos ordenadamente con el centinela.

In [13]:
class MonitorEventos(threading.Thread):
    # Único consumidor de la cola de eventos: es el dueño exclusivo de la salida por consola.
    def __init__(self, reloj: RelojSimulado, verbose: bool = True):
        super().__init__(name="Monitor", daemon=True)
        self.cola = queue.Queue()   # CANAL de comunicación: ya es thread-safe por dentro
        self.reloj = reloj
        self.verbose = verbose
        self.historial = []

    def publicar(self, quien: str, mensaje: str) -> None:
        # Lo llaman los hilos camión (productores). No los bloquea: la cola no tiene límite.
        self.cola.put((self.reloj.ahora(), quien, mensaje))

    def run(self) -> None:
        while True:
            evento = self.cola.get()     # se bloquea si la cola está vacía (espera pasiva, sin busy-wait)
            if evento is None:           # píldora de veneno -> fin del canal
                self.cola.task_done()
                break
            self.historial.append(evento)
            if self.verbose:
                t, quien, mensaje = evento
                print(f"[{formato_reloj(t)}] {quien:<10} | {mensaje}")
            self.cola.task_done()

    def detener(self) -> None:
        self.cola.put(None)   # se encola DETRÁS de todos los eventos pendientes
        self.join()           # esperamos a que el monitor termine de vaciar la cola

## 5. Sincronización por exclusión mutua: las plantas

Cada planta tiene dos recursos independientes de capacidad 1, tal como pide el enunciado
("cada planta puede manejar como máximo dos camiones a la vez, uno de carga y uno de descarga"):

```
        +------------ PLANTA -------------+
        |  zona_carga     (capacidad 1)   |
        |  zona_descarga  (capacidad 1)   |
        +---------------------------------+
```

Son `Semaphore(1)`, es decir **semáforos binarios** usados como mutex. Un camión que llega y encuentra la
zona ocupada queda **bloqueado en el semáforo** (espera pasiva, no consume CPU) y el sistema operativo lo
despierta cuando se libera. Eso es literalmente "el primer camión disponible ingresa *si está libre la zona
de carga*".

Los usamos con `with`, que garantiza el `release()` **aunque el bloque lance una excepción**. Un `acquire()`
sin su `release()` deja el recurso tomado para siempre y cuelga a todos los demás camiones.

### La bolsa de viajes: sección crítica con `Lock`

`viajes_pendientes -= 1` **no es atómico**: son leer, restar y escribir. Si dos camiones lo hacen a la vez
pueden leer el mismo valor y descontar un solo viaje por dos que salieron (una **condición de carrera**
clásica, del tipo *lost update*). Peor aún, el patrón que necesitamos es *comprobar-y-actuar* ("si hay
viajes, tomá uno"), que debe ser **una sola operación indivisible**.

Por eso `tomar_viaje()` hace la comprobación y el decremento **dentro del mismo `Lock`**.

> **Diferencia `Lock` vs `Semaphore(1)`:** un `Lock` es un mutex (conceptualmente tiene dueño y protege
> datos); un semáforo es un contador de permisos y puede liberarlo un hilo distinto del que lo tomó.
> Acá usamos `Lock` para proteger **datos** y `Semaphore` para modelar **recursos físicos** con cupo.

In [14]:
class Planta:
    # Una planta = 2 recursos exclusivos + la bolsa de viajes que se despachan desde acá.
    def __init__(self, nombre: str, viajes: int):
        self.nombre = nombre
        # SINCRONIZACIÓN: recursos físicos con cupo 1 (semáforos binarios).
        self.zona_carga    = threading.Semaphore(1)
        self.zona_descarga = threading.Semaphore(1)
        # SINCRONIZACIÓN: mutex que protege el estado compartido de abajo.
        self._mutex = threading.Lock()
        self._viajes_pendientes = viajes   # dato compartido y mutable: sólo se toca con el mutex tomado

    def tomar_viaje(self) -> bool:
        # Comprobar-y-actuar ATÓMICO: sin el lock, dos camiones podrían llevarse el mismo último viaje.
        with self._mutex:
            if self._viajes_pendientes > 0:
                self._viajes_pendientes -= 1
                return True
            return False

    @property
    def viajes_pendientes(self) -> int:
        with self._mutex:
            return self._viajes_pendientes

    def __repr__(self) -> str:
        return f"<Planta {self.nombre}: {self.viajes_pendientes} viajes pendientes>"

## 6. Semáforo contador: la estación de servicio

Los surtidores son el caso donde el semáforo se usa **como lo concibió Dijkstra**: un contador de permisos.
`BoundedSemaphore(2)` admite dos camiones simultáneos y bloquea al tercero hasta que alguno termine de
cargar combustible.

Elegimos la variante **`Bounded`** porque lanza `ValueError` si alguien hace un `release()` de más: es una
red de seguridad contra el bug de "liberar un recurso que no tomé", que en un semáforo común pasaría
inadvertido e inventaría surtidores que no existen.

También definimos un contador global de viajes completados, otra vez protegido por un `Lock`: es la variable
compartida más obvia del programa y la que usamos para **verificar un invariante** al final (se completaron
exactamente X viajes, ni uno más ni uno menos).

In [15]:
class EstacionDeServicio:
    # Recurso compartido con cupo > 1: el semáforo contador clásico.
    def __init__(self, cantidad_surtidores: int = CANT_SURTIDORES):
        self.surtidores = threading.BoundedSemaphore(cantidad_surtidores)


class ContadorViajes:
    # Contador compartido por todos los hilos: cada incremento es una sección crítica.
    def __init__(self):
        self._n = 0
        self._mutex = threading.Lock()

    def incrementar(self) -> int:
        with self._mutex:          # sin este lock, "self._n += 1" pierde incrementos
            self._n += 1
            return self._n

    @property
    def total(self) -> int:
        with self._mutex:
            return self._n

## 7. El hilo Camión

Cada camión es un `Thread` con su propio estado **privado** (ubicación, velocidad del chofer, viajes
hechos). Ese estado no necesita sincronización: **es local al hilo**, nadie más lo escribe. Sólo se
sincroniza lo que se comparte.

### Ciclo de vida

```
    +--> ¿hay viaje pendiente en mi planta?
    |         | sí                          | no
    |         v                             v
    |   [zona_carga]  2 h          ¿hay viajes en la otra planta?
    |         |                       | sí             | no
    |         v                       v                v
    |   (sólo Fernández)         viaja VACÍO       FIN del hilo
    |   [surtidor]    1 h
    |         |
    |         v
    |   en ruta 18-24 h  (según el chofer)
    |         |
    |         v
    |   [zona_descarga del destino]  2 h
    +---------+
```

### Tres decisiones de diseño que evitan interbloqueo (*deadlock*)

Un interbloqueo necesita las cuatro condiciones de Coffman a la vez: exclusión mutua, retención y espera,
no apropiación y **espera circular**. Acá:

1. **Nunca se retienen dos recursos a la vez.** El camión suelta la zona de carga *antes* de pedir el
   surtidor, y suelta el surtidor *antes* de salir a la ruta. Se rompe "retención y espera", así que no
   puede formarse una cadena de espera circular.
2. **La zona de descarga se pide en el destino, ya libre de cualquier otro recurso.** Un camión que espera
   para descargar no está bloqueando ninguna zona de carga.
3. **El `with` garantiza la liberación**, incluso ante una excepción.

### Sincronización por barrera

`threading.Barrier(N)` hace que ningún camión arranque hasta que los N estén listos. Modela "los N camiones
empiezan en Bs. As." y, de paso, evita que el primer hilo creado le saque ventaja a los últimos (que el
`for ... start()` crea milisegundos después). Es una **sincronización por punto de encuentro**: todos se
bloquean hasta que llega el último y recién ahí se liberan juntos.

In [16]:
class Camion(threading.Thread):
    def __init__(self, id_camion, plantas, estacion, monitor, reloj, barrera, contador, azar):
        super().__init__(name=f"Camión-{id_camion}")
        self.id = id_camion
        # Referencias a los objetos COMPARTIDOS (memoria compartida, ya protegida por dentro):
        self.plantas = plantas
        self.estacion = estacion
        self.monitor = monitor
        self.reloj = reloj
        self.barrera = barrera
        self.contador = contador
        # Estado PRIVADO del hilo: no requiere sincronización de ningún tipo.
        self.horas_por_tramo = azar.uniform(VIAJE_MIN_H, VIAJE_MAX_H)   # "depende del chofer"
        self.ubicacion = TAPIALES      # "los N camiones empiezan en Bs. As."
        self.viajes_hechos = 0
        self.tramos_vacio = 0

    def _avisar(self, mensaje: str) -> None:
        # COMUNICACIÓN: el camión no imprime, publica un mensaje en el canal.
        self.monitor.publicar(self.name, mensaje)

    @staticmethod
    def _otra_planta(nombre: str) -> str:
        return FERNANDEZ if nombre == TAPIALES else TAPIALES

    def run(self) -> None:
        self._avisar(f"listo en {self.ubicacion} (este chofer tarda {self.horas_por_tramo:.1f} h por tramo)")
        self.barrera.wait()            # SINCRONIZACIÓN: largada simultánea de los N camiones
        self._avisar("inicia la jornada")

        while True:
            origen  = self.plantas[self.ubicacion]
            destino = self.plantas[self._otra_planta(self.ubicacion)]

            if origen.tomar_viaje():                    # sección crítica: me asigno un viaje
                self._avisar(f"espera por CARGA en {origen.nombre}")
                with origen.zona_carga:                 # se bloquea si hay otro camión cargando
                    self._avisar(f"CARGANDO en {origen.nombre}")
                    self.reloj.dormir(HORAS_CARGA)
                # Libera la zona de carga ANTES de pedir el surtidor: nunca retiene dos recursos.

                if origen.nombre == FERNANDEZ:
                    self._avisar("espera SURTIDOR de Diesel")
                    with self.estacion.surtidores:      # semáforo contador: sólo 2 camiones a la vez
                        self._avisar("cargando COMBUSTIBLE")
                        self.reloj.dormir(HORAS_COMBUSTIBLE)
                    self._avisar("tanque lleno")

                carga = "harina" if origen.nombre == TAPIALES else "carbón"
                self._avisar(f"PARTE CARGADO con {carga} hacia {destino.nombre}")
                self.reloj.dormir(self.horas_por_tramo)  # en ruta: libera CPU y GIL

                self._avisar(f"llegó a {destino.nombre}, espera por DESCARGA")
                with destino.zona_descarga:              # se bloquea si hay otro descargando
                    self._avisar(f"DESCARGANDO en {destino.nombre}")
                    self.reloj.dormir(HORAS_DESCARGA)

                self.ubicacion = destino.nombre
                self.viajes_hechos += 1
                total = self.contador.incrementar()      # sección crítica
                self._avisar(f"viaje completado (total de la flota: {total})")

            elif destino.viajes_pendientes > 0:
                # No hay carga acá pero sí en la otra planta: se reposiciona en vacío.
                self._avisar(f"sin carga en {origen.nombre}: viaja VACÍO a {destino.nombre}")
                self.reloj.dormir(self.horas_por_tramo)
                self.ubicacion = destino.nombre
                self.tramos_vacio += 1

            else:
                self._avisar(f"no quedan viajes: FIN DE JORNADA en {origen.nombre}")
                break                                    # termina el hilo

## 8. Orquestación de la simulación

El hilo principal actúa de coordinador:

1. Crea los objetos compartidos (plantas, estación, contador, monitor).
2. Lanza los N hilos camión con `start()`.
3. Se **bloquea en `join()`** hasta que cada camión termina. `join()` es la sincronización de terminación:
   garantiza que, cuando el hilo principal siga, ya no hay nadie tocando el estado compartido, así que leer
   los resultados después del `join()` es seguro sin ningún candado.
4. Cierra el canal de eventos con el centinela.

Los X viajes se reparten entre las dos plantas (la mitad de ida, la mitad de vuelta) y la simulación termina
cuando **ninguna** de las dos tiene viajes pendientes, tal como pide el enunciado.

In [17]:
def simular(n_camiones: int, x_viajes: int, escala: float = 0.05,
            verbose: bool = True, semilla: int = 42) -> dict:
    azar = random.Random(semilla)     # semilla fija: los choferes son comparables entre corridas
    reloj = RelojSimulado(escala)

    # --- Objetos compartidos ---
    monitor = MonitorEventos(reloj, verbose)
    monitor.start()
    plantas = {
        TAPIALES:  Planta(TAPIALES,  (x_viajes + 1) // 2),   # viajes de ida (harina)
        FERNANDEZ: Planta(FERNANDEZ, x_viajes // 2),         # viajes de vuelta (carbón)
    }
    estacion = EstacionDeServicio()
    contador = ContadorViajes()
    barrera  = threading.Barrier(n_camiones)

    camiones = [Camion(i + 1, plantas, estacion, monitor, reloj, barrera, contador, azar)
                for i in range(n_camiones)]

    t_inicio = time.perf_counter()
    for c in camiones:
        c.start()      # a partir de acá los N camiones corren junto con el hilo principal
    for c in camiones:
        c.join()       # el hilo principal espera a TODOS
    horas = (time.perf_counter() - t_inicio) / escala

    monitor.detener()  # centinela + join del consumidor

    return {
        "camiones": n_camiones,
        "viajes_pedidos": x_viajes,
        "viajes_completados": contador.total,
        "horas": horas,
        "dias": horas / 24,
        "por_camion": {c.name: c.viajes_hechos for c in camiones},
        "tramos_vacio": sum(c.tramos_vacio for c in camiones),
        "eventos": len(monitor.historial),
    }

## 9. Corrida de demostración: 3 camiones, 8 viajes

La traza muestra lo que pide el enunciado: **el estado de cada camión** (espera por carga, parte cargado,
carga combustible, descarga) y, al final, **el tiempo total en días**.

Al leerla conviene mirar tres cosas:

- Las líneas de distintos camiones **se entrelazan**: eso es concurrencia real, no una simulación por turnos.
- Entre `espera por CARGA` y `CARGANDO` a veces pasan horas: ahí el camión estuvo **bloqueado en el
  semáforo** porque la zona estaba ocupada. Nunca aparecen dos `CARGANDO` simultáneos en la misma planta:
  eso es la **exclusión mutua** funcionando.
- Los sellos de tiempo salen **ordenados** aunque los produzcan hilos distintos, porque la cola preserva el
  orden de llegada y hay un solo consumidor.

In [18]:
resultado = simular(n_camiones=3, x_viajes=8, escala=0.05, verbose=True)

print()
print("=" * 72)
print(f"Camiones: {resultado['camiones']}   |   Viajes solicitados: {resultado['viajes_pedidos']}")
print(f"Viajes completados: {resultado['viajes_completados']}")
print(f"Tiempo total: {resultado['horas']:.1f} horas  =  {resultado['dias']:.2f} días")
print(f"Reparto de viajes: {resultado['por_camion']}")
print(f"Tramos recorridos en vacío: {resultado['tramos_vacio']}")
print("=" * 72)

# VERIFICACIÓN DE INVARIANTES: en concurrencia no alcanza con que "parezca" que funcionó.
assert resultado["viajes_completados"] == resultado["viajes_pedidos"], "se perdieron o duplicaron viajes"
assert sum(resultado["por_camion"].values()) == resultado["viajes_pedidos"], "el contador no cierra"
print("OK: se completaron exactamente los viajes pedidos, sin pérdidas ni duplicados.")

[Día 1 00:02] Camión-1   | listo en Tapiales (este chofer tarda 21.8 h por tramo)
[Día 1 00:02] Camión-2   | listo en Tapiales (este chofer tarda 18.2 h por tramo)
[Día 1 00:03] Camión-3   | listo en Tapiales (este chofer tarda 19.7 h por tramo)
[Día 1 00:03] Camión-3   | inicia la jornada
[Día 1 00:03] Camión-3   | espera por CARGA en Tapiales
[Día 1 00:03] Camión-3   | CARGANDO en Tapiales
[Día 1 00:03] Camión-2   | inicia la jornada
[Día 1 00:03] Camión-2   | espera por CARGA en Tapiales
[Día 1 00:03] Camión-1   | inicia la jornada
[Día 1 00:03] Camión-1   | espera por CARGA en Tapiales
[Día 1 02:03] Camión-3   | PARTE CARGADO con harina hacia Fernández
[Día 1 02:03] Camión-2   | CARGANDO en Tapiales
[Día 1 04:03] Camión-2   | PARTE CARGADO con harina hacia Fernández
[Día 1 04:03] Camión-1   | CARGANDO en Tapiales
[Día 1 06:04] Camión-1   | PARTE CARGADO con harina hacia Fernández
[Día 1 21:42] Camión-3   | llegó a Fernández, espera por DESCARGA
[Día 1 21:42] Camión-3   | DESCARGAND

### Qué prueban los `assert`

Si el decremento de `viajes_pendientes` no estuviera protegido por el `Lock`, dos camiones podrían tomar el
mismo último viaje: saldrían 9 camiones cargados para 8 viajes y el invariante fallaría. El `assert` es la
forma de detectar una condición de carrera que, por su naturaleza, **puede no manifestarse en cada corrida**.
Ése es el rasgo más incómodo de los errores de concurrencia: no son reproducibles a voluntad, así que se
combaten con diseño correcto e invariantes, no con depuración a mano.

**No determinismo:** si volvés a ejecutar la celda anterior, los tiempos cambian levemente y el reparto de
viajes entre camiones puede cambiar también, aunque la semilla sea fija. El planificador del sistema
operativo decide a quién despierta primero cuando se libera un semáforo, y eso el programa no lo controla.
Lo que sí está garantizado por construcción es el invariante: **siempre** se completan X viajes.

**Equidad (*fairness*) e inanición:** los semáforos de Python no garantizan orden FIFO. En teoría un camión
podría quedar postergado indefinidamente (**inanición**) si otros le ganan siempre el semáforo. En la
práctica la implementación despierta en orden de llegada y las esperas son larguísimas comparadas con un
cambio de contexto, pero si el enunciado exigiera estrictamente "el primero que llegó entra primero" habría
que implementar una cola explícita de turnos en vez de apoyarse en el semáforo.

**Una carrera benigna que quedó a la vista:** en el `elif`, el camión consulta `destino.viajes_pendientes`
y después viaja 20 horas. Al llegar, otro camión puede haberse llevado ese viaje, y entonces viajó en vacío
en vano. Es un *comprobar-y-actuar* que deliberadamente **no** hicimos atómico, porque la acción dura horas
y bloquear la planta todo ese tiempo sería peor. El programa sigue siendo correcto (el camión simplemente
vuelve a evaluar y termina), pero muestra que la atomicidad se decide caso por caso.

## 10. La pregunta del negocio: ¿cuántos camiones conviene asignar?

Ahora usamos el simulador para lo que la empresa lo pidió. Corremos la simulación variando N y medimos el
tiempo total para completar los mismos X viajes.

Como el resultado es no determinista (velocidad de cada chofer + decisiones del planificador), una sola
corrida por cada valor de N no alcanza: hacemos **varias réplicas con distintas semillas y promediamos**.
Eso también es parte de trabajar con concurrencia: los resultados son distribuciones, no números exactos.

Bajamos la escala de tiempo y apagamos la traza para que el barrido sea rápido (tarda ~30 segundos).

In [ ]:
X_VIAJES = 12
SEMILLAS = (1, 7, 42)
RANGO_N  = range(1, 9)

resultados_barrido = []
for n in RANGO_N:
    corridas = [simular(n, X_VIAJES, escala=0.01, verbose=False, semilla=s)["dias"] for s in SEMILLAS]
    resultados_barrido.append({"n": n, "dias": statistics.mean(corridas),
                               "min": min(corridas), "max": max(corridas)})

print(f"{X_VIAJES} viajes — promedio de {len(SEMILLAS)} corridas por cada N\n")
print(f"{'N':>2} | {'días':>6} | {'mejora':>7} | perfil")
print("-" * 58)
anterior = None
for r in resultados_barrido:
    mejora = "—" if anterior is None else f"{(anterior - r['dias']) / anterior * 100:5.1f} %"
    barra = "#" * max(1, round(r["dias"] * 3))
    print(f"{r['n']:>2} | {r['dias']:6.2f} | {mejora:>7} | {barra}")
    anterior = r["dias"]

### Lectura del barrido

Se ve el efecto que la empresa necesitaba cuantificar: **agregar camiones deja de servir**. Al principio
duplicar la flota casi divide a la mitad el tiempo (los camiones trabajan en paralelo sin estorbarse), pero
a partir de cierto punto pasan más tiempo **bloqueados esperando** una zona de carga o un surtidor que
rodando.

Es exactamente la **ley de Amdahl** aplicada a la operación: las partes serializadas del proceso —las zonas
de capacidad 1 y los 2 surtidores— ponen un piso al tiempo total que ningún número de camiones puede bajar.
El cuello de botella deja de ser la flota y pasa a ser la infraestructura de las plantas.

**La curva no baja de forma suave, baja a escalones.** Con 12 viajes (6 por planta) el tiempo total lo fija
la cantidad de *rondas* que tiene que hacer el camión más cargado, y una ronda dura ≈25 h haya uno o cinco
camiones. Por eso pasar de 4 a 5 camiones casi no mejora (el quinto no elimina ninguna ronda, sólo agrega
cola en la zona de carga), y en cambio pasar a 6 sí produce un salto: recién ahí cada camión hace
exactamente una ida y una vuelta y desaparece la segunda ronda. Es el mismo fenómeno de **granularidad**
que aparece al repartir tareas entre hilos: si el trabajo no divide justo, agregar trabajadores no baja el
tiempo del más cargado.

In [ ]:
import matplotlib.pyplot as plt

TINTA       = "#0b0b0b"
TINTA_SUAVE = "#52514e"
SUPERFICIE  = "#fcfcfb"
SERIE       = "#2a78d6"

ns   = [r["n"] for r in resultados_barrido]
dias = [r["dias"] for r in resultados_barrido]

fig, ax = plt.subplots(figsize=(7.5, 4.4))
fig.patch.set_facecolor(SUPERFICIE)
ax.set_facecolor(SUPERFICIE)

ax.plot(ns, dias, color=SERIE, linewidth=2, marker="o", markersize=7,
        markerfacecolor=SERIE, markeredgecolor=SUPERFICIE, markeredgewidth=2, zorder=3)

# Codo de la curva: el último N que todavía aporta una mejora relevante (>= 10 %).
codo = ns[0]
for i in range(1, len(dias)):
    if (dias[i - 1] - dias[i]) / dias[i - 1] >= 0.10:
        codo = ns[i]
idx = ns.index(codo)
ax.annotate(f"{codo} camiones · {dias[idx]:.1f} días\nsumar flota casi no mejora",
            xy=(codo, dias[idx]), xytext=(codo + 0.4, dias[idx] + max(dias) * 0.22),
            color=TINTA, fontsize=9.5, va="bottom",
            arrowprops=dict(arrowstyle="-", color=TINTA_SUAVE, linewidth=1))

ax.set_title(f"Tiempo para completar {X_VIAJES} viajes según el tamaño de la flota",
             color=TINTA, fontsize=12.5, pad=14, loc="left")
ax.set_xlabel("Camiones asignados (N)", color=TINTA_SUAVE, fontsize=10)
ax.set_ylabel("Días", color=TINTA_SUAVE, fontsize=10)
ax.set_xticks(ns)
ax.set_ylim(0, max(dias) * 1.2)
ax.grid(axis="y", color="#d9d8d3", linewidth=0.6)
ax.set_axisbelow(True)
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
for lado in ("left", "bottom"):
    ax.spines[lado].set_color("#d9d8d3")
ax.tick_params(colors=TINTA_SUAVE, length=0)

plt.tight_layout()
plt.show()

## 11. Conclusión para Carga Criolla S.A.

- El tiempo total cae fuerte con los primeros camiones y después **se aplana**: sumar unidades más allá del
  codo de la curva agrega costo fijo (camión, chofer, combustible) sin acortar la operación.
- El techo lo imponen las **zonas de carga y descarga de capacidad 1**. Si la empresa quisiera bajar todavía
  más el tiempo, la inversión rentable no es comprar camiones sino **habilitar una segunda zona de carga**
  (en el modelo: cambiar `Semaphore(1)` por `Semaphore(2)` en `Planta` y volver a correr el barrido).
- Los surtidores (capacidad 2) casi no generan cola porque repostar es corto (1 h) frente al viaje (≈21 h):
  **el recurso escaso no es el que uno intuye, es el que la simulación muestra**.

> El cuaderno se puede reejecutar cambiando `X_VIAJES` o la capacidad de las zonas para responder otras
> preguntas del negocio sin tocar la lógica de sincronización.

## 12. Resumen de los conceptos aplicados

| Concepto | Dónde aparece en este cuaderno |
|---|---|
| **Hilo** (`threading.Thread`) | Cada camión y el monitor de eventos |
| **Estado privado del hilo** | Ubicación, velocidad del chofer y contadores propios de cada camión: no necesitan sincronización |
| **Datos inmutables compartidos** | Las constantes del enunciado: se comparten sin protección porque nadie las escribe |
| **Comunicación por memoria compartida** | Plantas, estación de servicio y contador global de viajes |
| **Comunicación por paso de mensajes** | `queue.Queue` de eventos entre camiones (productores) y monitor (consumidor) |
| **Productor–consumidor** | El patrón del monitor de eventos |
| **Píldora de veneno / centinela** | El `None` que cierra el canal y termina al monitor |
| **Sección crítica** | El cuerpo de `tomar_viaje()` y de `ContadorViajes.incrementar()` |
| **Exclusión mutua** (`Lock`) | Protección de `viajes_pendientes` y del contador de viajes |
| **Semáforo binario** (`Semaphore(1)`) | Zonas de carga y descarga de cada planta |
| **Semáforo contador** (`BoundedSemaphore(2)`) | Los dos surtidores de Diesel de Fernández |
| **Barrera** (`Barrier(N)`) | Largada simultánea de los N camiones |
| **Sincronización de terminación** (`join`) | El hilo principal esperando a los camiones y al monitor |
| **Espera pasiva (bloqueo)** | `acquire()` de los semáforos y `queue.get()`: sin *busy-waiting* |
| **Condición de carrera** | El `-= 1` y el comprobar-y-actuar que el `Lock` vuelve atómicos |
| **Prevención de interbloqueo** | Nunca se retienen dos recursos a la vez; `with` garantiza la liberación |
| **Inanición y equidad** | Sección 9: los semáforos de Python no garantizan FIFO |
| **No determinismo** | Los tiempos y el reparto cambian entre corridas; los invariantes no |
| **GIL** | Por qué los hilos sirven acá (esperas) y no servirían para cálculo puro |
| **Ley de Amdahl** | El piso de tiempo que imponen los recursos serializados (sección 10) |